# 🌧️ Urban Road Flood-Risk Prediction Using Graph Neural Networks

Predict Low, Medium, or High flood risk for every connected road section and rank which roads are likely to flood first.

> Educational synthetic city—not an emergency-routing system.

👉 **Open the interactive companion:** [https://urban-road-flood-gnn.streamlit.app](https://urban-road-flood-gnn.streamlit.app/?stage=start)

## Complete workflow

Create road graph → assign five node features → simulate storms → normalize adjacency → train GCN → map risk → rank roads → compare with isolated-road MLP.

## Interactive learning journey

- [Which Road Floods First?](https://urban-road-flood-gnn.streamlit.app/?stage=problem) — Node-Level Flood Prediction
- [Roads as a Connected Network](https://urban-road-flood-gnn.streamlit.app/?stage=graph) — Graph Representation
- [Five Measurements Per Road](https://urban-road-flood-gnn.streamlit.app/?stage=features) — Node Features
- [Virtual Urban Storms](https://urban-road-flood-gnn.streamlit.app/?stage=simulate) — Synthetic Graph Dataset
- [How Neighbour Influence Is Weighted](https://urban-road-flood-gnn.streamlit.app/?stage=adjacency) — Normalized Adjacency
- [Learning From This Road and Its Neighbours](https://urban-road-flood-gnn.streamlit.app/?stage=gcn) — Graph Convolutional Network
- [Learning Across Many Storms](https://urban-road-flood-gnn.streamlit.app/?stage=training) — Node Classification Training
- [Flooding Order and Main Reasons](https://urban-road-flood-gnn.streamlit.app/?stage=ranking) — Risk Ranking
- [The Urban Drainage Audit](https://urban-road-flood-gnn.streamlit.app/?stage=audit) — GNN vs Isolated-Road Baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow.keras import layers,Model
SEED=42;np.random.seed(SEED);tf.random.set_seed(SEED)
N_NODES,N_FEATURES,N_CLASSES=30,5,3
FEATURES=["rain_mm_hr","elevation_m","water_depth_cm","drainage_mm_hr","slope_pct"]

---
# 1. Which Road Floods First?
### Phase 1 of 6 · The Connected Flood Problem

## Part 1 · On the road network
During intense rainfall, water accumulates differently across roads because elevation, drainage, slope, and incoming flow vary.

## Part 2 · The engineering challenge
Monitoring current depth alone can miss a low road that is still shallow but is about to receive water from several neighbours.

## Part 3 · Where the AI comes in
Predict a flood probability for every road before depths become impassable, then rank the likely sequence.

**Civil Engineering:** Which Road Floods First? → **AI:** Node-Level Flood Prediction → `risk for every connected road section`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=problem](https://urban-road-flood-gnn.streamlit.app/?stage=problem)

## Part 4 · The technical explanation

The model returns a risk class and probability for every road node. Sorting High-risk probability produces the predicted flooding order.

## Part 5 · What you just built

**In the notebook:** Define node-level flood risk and the road-ranking output.

**Takeaway:** The earliest vulnerable road may not be the wettest road right now.

[Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Roads as a Connected Network](https://urban-road-flood-gnn.streamlit.app/?stage=graph) ▶

---
# 2. Roads as a Connected Network
### Phase 2 of 6 · Building the City Graph

## Part 1 · On the road network
Intersections and connecting road segments provide paths through which surface water and traffic effects propagate.

## Part 2 · The engineering challenge
A table of isolated rows discards which road can send water toward which neighbour.

## Part 3 · Where the AI comes in
Represent each road section as a node and hydraulic/physical connections as graph edges.

**Civil Engineering:** Roads as a Connected Network → **AI:** Graph Representation → `nodes = roads; edges = connections`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=graph](https://urban-road-flood-gnn.streamlit.app/?stage=graph)

## Part 4 · The technical explanation

In [ ]:
rng=np.random.default_rng(SEED);G=nx.random_geometric_graph(N_NODES,.31,seed=SEED)
while not nx.is_connected(G):G=nx.random_geometric_graph(N_NODES,.34,seed=int(rng.integers(9999)))
A=nx.to_numpy_array(G,dtype="float32");pos=nx.get_node_attributes(G,"pos")
nx.draw(G,pos,node_size=180,node_color="skyblue",edge_color="gray",with_labels=True);plt.title("Virtual 30-road network");plt.show()
print("Road nodes:",G.number_of_nodes(),"Connections:",G.number_of_edges())

## Part 5 · What you just built

**In the notebook:** Create a 30-node virtual road graph and adjacency matrix.

**Takeaway:** Connectivity is an engineering input, not merely a drawing.

◀ [Previous: Which Road Floods First?](https://urban-road-flood-gnn.streamlit.app/?stage=problem) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Five Measurements Per Road](https://urban-road-flood-gnn.streamlit.app/?stage=features) ▶

---
# 3. Five Measurements Per Road
### Phase 2 of 6 · Building the City Graph

## Part 1 · On the road network
Each road has rainfall intensity, elevation, current water depth, drainage capacity, and slope.

## Part 2 · The engineering challenge
Raw units differ, and slope direction affects whether a neighbour is upstream or downstream.

## Part 3 · Where the AI comes in
Store five normalized features on every node while the graph stores who can influence whom.

**Civil Engineering:** Five Measurements Per Road → **AI:** Node Features → `rain, elevation, depth, drainage, slope`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=features](https://urban-road-flood-gnn.streamlit.app/?stage=features)

## Part 4 · The technical explanation

In [ ]:
elevation=rng.uniform(96,112,N_NODES);drainage=rng.uniform(18,70,N_NODES);slope=rng.uniform(.2,4,N_NODES)
example=pd.DataFrame(dict(Road=np.arange(N_NODES),Elevation_m=elevation,Drainage_mm_hr=drainage,Slope_pct=slope));example.head()

## Part 5 · What you just built

**In the notebook:** Assemble an N×5 feature matrix and retain physical units for interpretation.

**Takeaway:** Node features describe the road; edges describe its context.

◀ [Previous: Roads as a Connected Network](https://urban-road-flood-gnn.streamlit.app/?stage=graph) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Virtual Urban Storms](https://urban-road-flood-gnn.streamlit.app/?stage=simulate) ▶

---
# 4. Virtual Urban Storms
### Phase 3 of 6 · Simulating Storms

## Part 1 · On the road network
A teaching city can be subjected to many storms while road elevations and drainage capacities remain fixed.

## Part 2 · The engineering challenge
Random labels would not demonstrate why the graph matters.

## Part 3 · Where the AI comes in
Simulate runoff, drainage removal, and elevation-directed neighbour transfer to create flood labels.

**Civil Engineering:** Virtual Urban Storms → **AI:** Synthetic Graph Dataset → `mass-balance-inspired runoff and neighbour inflow`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=simulate](https://urban-road-flood-gnn.streamlit.app/?stage=simulate)

## Part 4 · The technical explanation

In [ ]:
def storm_snapshot(rain):
 depth=np.clip(.045*rain-.035*drainage+.12*(105-elevation)+rng.normal(0,.8,N_NODES),0,None)
 # Water influence travels preferentially toward lower connected roads.
 incoming=np.zeros(N_NODES)
 for i,j in G.edges:
  if elevation[i]>elevation[j]:incoming[j]+=.16*depth[i]
  else:incoming[i]+=.16*depth[j]
 future=depth+incoming+.018*np.maximum(0,rain-drainage)
 labels=np.digitize(future,[7,14])
 X=np.column_stack([np.full(N_NODES,rain),elevation,depth,drainage,slope]).astype("float32")
 return X,labels,future,incoming
Xs,ys=[],[]
for _ in range(650):
 X,y,_,_=storm_snapshot(rng.uniform(20,145));Xs.append(X);ys.append(y)
Xs=np.array(Xs);ys=np.array(ys);print(Xs.shape,ys.shape,"class counts",np.bincount(ys.ravel()))

## Part 5 · What you just built

**In the notebook:** Generate hundreds of graph snapshots with node-level risk classes.

**Takeaway:** Synthetic labels should depend on both local conditions and connected upstream water.

◀ [Previous: Five Measurements Per Road](https://urban-road-flood-gnn.streamlit.app/?stage=features) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: How Neighbour Influence Is Weighted](https://urban-road-flood-gnn.streamlit.app/?stage=adjacency) ▶

---
# 5. How Neighbour Influence Is Weighted
### Phase 3 of 6 · Simulating Storms

## Part 1 · On the road network
Every road keeps its own information while also receiving evidence from connected sections.

## Part 2 · The engineering challenge
Simply summing neighbours makes high-degree intersections numerically dominate.

## Part 3 · Where the AI comes in
Add self-connections and degree-normalize the adjacency matrix before graph convolution.

**Civil Engineering:** How Neighbour Influence Is Weighted → **AI:** Normalized Adjacency → `Â = D⁻½(A+I)D⁻½`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=adjacency](https://urban-road-flood-gnn.streamlit.app/?stage=adjacency)

## Part 4 · The technical explanation

In [ ]:
A_self=A+np.eye(N_NODES,dtype="float32");degree=A_self.sum(1);D_inv=np.diag(1/np.sqrt(degree));A_norm=(D_inv@A_self@D_inv).astype("float32")
road=7;print("Road",road,"neighbors:",list(G.neighbors(road)));print("Self weight:",A_norm[road,road]);plt.imshow(A_norm,cmap="Blues");plt.colorbar();plt.title("Normalized adjacency Â");plt.show()

## Part 5 · What you just built

**In the notebook:** Calculate normalized adjacency and inspect one road's neighbourhood.

**Takeaway:** Normalization balances self-information and neighbour messages.

◀ [Previous: Virtual Urban Storms](https://urban-road-flood-gnn.streamlit.app/?stage=simulate) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning From This Road and Its Neighbours](https://urban-road-flood-gnn.streamlit.app/?stage=gcn) ▶

---
# 6. Learning From This Road and Its Neighbours
### Phase 4 of 6 · Learning From Neighbours

## Part 1 · On the road network
Road C may be low and poorly drained while Road B and Road E are sending it additional water.

## Part 2 · The engineering challenge
An MLP sees Road C alone and cannot distinguish identical local readings embedded in different networks.

## Part 3 · Where the AI comes in
GCN layers aggregate connected-node features, transform them, and pass updated representations through the graph.

**Civil Engineering:** Learning From This Road and Its Neighbours → **AI:** Graph Convolutional Network → `H′ = ReLU(ÂHW)`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=gcn](https://urban-road-flood-gnn.streamlit.app/?stage=gcn)

## Part 4 · The technical explanation

In [ ]:
class GraphConv(layers.Layer):
 def __init__(self,units,activation=None):super().__init__();self.units=units;self.activation=tf.keras.activations.get(activation)
 def build(self,input_shape):self.w=self.add_weight(shape=(input_shape[-1],self.units),initializer="glorot_uniform")
 def call(self,inputs):
  x,a=inputs;h=tf.matmul(x,self.w);out=tf.einsum("ij,bjk->bik",a,h);return self.activation(out) if self.activation else out
xin=layers.Input((N_NODES,N_FEATURES));ain=layers.Input((N_NODES,N_NODES));h=GraphConv(32,"relu")([xin,ain]);h=layers.Dropout(.12)(h);h=GraphConv(24,"relu")([h,ain]);out=layers.Dense(N_CLASSES,activation="softmax")(h);gcn_model=Model([xin,ain],out);gcn_model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"]);gcn_model.summary()

## Part 5 · What you just built

**In the notebook:** Build two pure-TensorFlow graph-convolution layers and a node classifier.

**Takeaway:** The GNN learns from conditions and connections simultaneously.

◀ [Previous: How Neighbour Influence Is Weighted](https://urban-road-flood-gnn.streamlit.app/?stage=adjacency) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Learning Across Many Storms](https://urban-road-flood-gnn.streamlit.app/?stage=training) ▶

---
# 7. Learning Across Many Storms
### Phase 4 of 6 · Learning From Neighbours

## Part 1 · On the road network
Each simulated storm labels every road as Low, Medium, or High risk.

## Part 2 · The engineering challenge
Most roads may remain safe, so unweighted accuracy can ignore the rare high-risk nodes.

## Part 3 · Where the AI comes in
Train on graph snapshots, monitor validation loss, and report class-specific recall.

**Civil Engineering:** Learning Across Many Storms → **AI:** Node Classification Training → `weighted cross-entropy + validation`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=training](https://urban-road-flood-gnn.streamlit.app/?stage=training)

## Part 4 · The technical explanation

In [ ]:
idx=np.arange(len(Xs));tr,tmp=train_test_split(idx,test_size=.30,random_state=SEED);va,te=train_test_split(tmp,test_size=.50,random_state=SEED)
scaler=StandardScaler().fit(Xs[tr].reshape(-1,N_FEATURES));scale=lambda x:scaler.transform(x.reshape(-1,N_FEATURES)).reshape(x.shape);Xtr,Xva,Xte=scale(Xs[tr]),scale(Xs[va]),scale(Xs[te]);Atr=np.repeat(A_norm[None],len(tr),0);Ava=np.repeat(A_norm[None],len(va),0);Ate=np.repeat(A_norm[None],len(te),0)
weights=np.array([1,1.4,1.8]);sample_weight=weights[ys[tr]];history=gcn_model.fit([Xtr,Atr],ys[tr],sample_weight=sample_weight,validation_data=([Xva,Ava],ys[va]),epochs=40,batch_size=24,verbose=0)
plt.plot(history.history["loss"],label="train");plt.plot(history.history["val_loss"],label="validation");plt.grid(alpha=.2);plt.legend();plt.show()

## Part 5 · What you just built

**In the notebook:** Train the GCN on balanced/weighted node labels.

**Takeaway:** A flood model must be judged on high-risk roads, not only the safe majority.

◀ [Previous: Learning From This Road and Its Neighbours](https://urban-road-flood-gnn.streamlit.app/?stage=gcn) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: Flooding Order and Main Reasons](https://urban-road-flood-gnn.streamlit.app/?stage=ranking) ▶

---
# 8. Flooding Order and Main Reasons
### Phase 5 of 6 · Ranking Vulnerable Roads

## Part 1 · On the road network
Authorities need a prioritized list for inspection, traffic diversion, and drainage response.

## Part 2 · The engineering challenge
A probability without network context gives little insight into why one road ranks above another.

## Part 3 · Where the AI comes in
Sort high-risk probabilities and display local elevation/drainage plus connected upstream-water evidence.

**Civil Engineering:** Flooding Order and Main Reasons → **AI:** Risk Ranking → `sort node high-risk probability`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=ranking](https://urban-road-flood-gnn.streamlit.app/?stage=ranking)

## Part 4 · The technical explanation

In [ ]:
X_new,y_true,future,incoming=storm_snapshot(85);probs=gcn_model.predict([scale(X_new[None]),A_norm[None]],verbose=0)[0];order=np.argsort(probs[:,2])[::-1]
print("AI FLOOD PREDICTION")
for rank,node in enumerate(order[:5],1):print(f"{rank}. Road {node:02d} — high-risk probability {probs[node,2]:.1%}")
top=order[0];print("\nFirst road likely to flood: ROAD",top);print("Main reasons:",f"elevation {elevation[top]:.1f} m, drainage {drainage[top]:.1f} mm/hr, upstream contribution {incoming[top]:.1f} cm")
colors=probs[:,2];nx.draw(G,pos,node_color=colors,cmap="RdYlGn_r",vmin=0,vmax=1,with_labels=True,node_size=230);plt.title("Predicted high-flood-risk probability");plt.show()

## Part 5 · What you just built

**In the notebook:** Map probabilities, print the top-five order, and explain the leading road.

**Takeaway:** Ranking converts thirty node predictions into an actionable inspection priority.

◀ [Previous: Learning Across Many Storms](https://urban-road-flood-gnn.streamlit.app/?stage=training) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start) &nbsp;|&nbsp; [Next: The Urban Drainage Audit](https://urban-road-flood-gnn.streamlit.app/?stage=audit) ▶

---
# 9. The Urban Drainage Audit
### Phase 6 of 6 · Civil Engineering Audit

## Part 1 · On the road network
A graph model is justified only if connectivity improves decisions over a model that sees each road independently.

## Part 2 · The engineering challenge
Synthetic topology and simplified surface flow may not transfer to a real city's drains, kerbs, inlets, terrain, and blockages.

## Part 3 · Where the AI comes in
Compare against an MLP baseline, remove edges as an ablation, inspect high-risk misses, and state mapping limits.

**Civil Engineering:** The Urban Drainage Audit → **AI:** GNN vs Isolated-Road Baseline → `confusion matrix, high-risk recall, ablation`

> 🎬 **See this illustrated and interactive:** [https://urban-road-flood-gnn.streamlit.app/?stage=audit](https://urban-road-flood-gnn.streamlit.app/?stage=audit)

## Part 4 · The technical explanation

In [ ]:
pred=np.argmax(gcn_model.predict([Xte,Ate],verbose=0),axis=2);print(classification_report(ys[te].ravel(),pred.ravel(),target_names=["Low","Medium","High"]));ConfusionMatrixDisplay(confusion_matrix(ys[te].ravel(),pred.ravel()),display_labels=["Low","Medium","High"]).plot(cmap="Blues");plt.show()
# Edge-removal ablation: same trained model, isolated self-only graph.
isolated=np.repeat(np.eye(N_NODES,dtype="float32")[None],len(te),0);isolated_pred=np.argmax(gcn_model.predict([Xte,isolated],verbose=0),axis=2)
print("GCN high-risk recall:",((pred==2)&(ys[te]==2)).sum()/(ys[te]==2).sum());print("Edges removed high-risk recall:",((isolated_pred==2)&(ys[te]==2)).sum()/(ys[te]==2).sum())
print("Limitations: synthetic topology, simplified surface transfer, no inlet/pipe capacity model, no terrain raster, no blockage, no uncertainty ensemble, no traffic or emergency integration.")

## Part 5 · What you just built

**In the notebook:** Report metrics, connectivity benefit, uncertainty, and real-data requirements.

**Takeaway:** The graph must earn its complexity through measurable neighbour-information value.

◀ [Previous: Flooding Order and Main Reasons](https://urban-road-flood-gnn.streamlit.app/?stage=ranking) &nbsp;|&nbsp; [Project overview](https://urban-road-flood-gnn.streamlit.app/?stage=start)

---
# Final engineering conclusion

The GCN predicts every road using its own five measurements and messages from connected neighbours. Flood-order ranking converts node probabilities into inspection priorities, while the edge-removal audit tests whether connectivity genuinely adds value.